In [1]:
!pip3 install wikipedia pandas tqdm

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11785 sha256=3e477b6b62a13e5fda4642349e1403e0f562f741989f8520cbf3b9643dfc4167
  Stored in directory: c:\users\sam\appdata\local\pip\cache\wheels\79\1d\c8\b64e19423cc5a2a339450ea5d145e7c8eb3d4aa2b150cde33b
Successfully built wikipedia


In [2]:

import os
import time
import requests
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# =====================================================
# Configuration
# =====================================================

OUTPUT_DIR = Path("AI_Knowledge_Base")
OUTPUT_DIR.mkdir(exist_ok=True)

HEADERS = {
    "User-Agent": "AIML-RAG-Dataset-Builder/1.0"
}

SEARCH_LIMIT = 10
MAX_RETRIES = 3
SLEEP_TIME = 1

# =====================================================
# Seed Keywords
# =====================================================

KEYWORDS = [

    "Artificial Intelligence",
    "Machine Learning",
    "Deep Learning",
    "Large Language Model",
    "Natural Language Processing",
    "Data Science",
    "Computer Vision",
    "Generative AI",
    "Transformer",
    "Neural Network",
    "Prompt Engineering",
    "Retrieval Augmented Generation",
    "Semantic Search",
    "Vector Database",
    "Feature Engineering"

]

# =====================================================
# Search Wikipedia
# =====================================================

def search_articles(keyword, limit=10):

    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": keyword,
        "srlimit": limit,
        "format": "json"
    }

    response = requests.get(
        url,
        params=params,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    titles = []

    for article in data["query"]["search"]:
        titles.append(article["title"])

    return titles

# =====================================================
# Download Wikipedia Article
# =====================================================

def download_article(title):

    url = "https://en.wikipedia.org/w/api.php"

    params = {

        "action": "query",
        "format": "json",
        "titles": title,
        "redirects": 1,
        "prop": "extracts|info",
        "inprop": "url",
        "explaintext": 1

    }

    response = requests.get(
        url,
        params=params,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    page = list(data["query"]["pages"].values())[0]

    if "missing" in page:
        return None

    return {

        "title": page["title"],
        "text": page.get("extract", ""),
        "url": page.get("fullurl", "")

    }

# =====================================================
# Safe filename
# =====================================================

def clean_filename(name):

    invalid = '<>:"/\\|?*'

    for ch in invalid:
        name = name.replace(ch, "_")

    return name

# =====================================================
# Main Download Loop
# =====================================================

metadata = []
errors = []

visited = set()

print("="*60)
print("Searching Wikipedia")
print("="*60)

candidate_titles = []

for keyword in KEYWORDS:

    print(f"\nSearching : {keyword}")

    try:

        results = search_articles(keyword, SEARCH_LIMIT)

        candidate_titles.extend(results)

        print(f"Found {len(results)} articles")

    except Exception as e:

        print(e)

candidate_titles = sorted(set(candidate_titles))

print("\n")
print("="*60)
print(f"Total Unique Articles Found : {len(candidate_titles)}")
print("="*60)

# =====================================================
# Download
# =====================================================

for title in tqdm(candidate_titles):

    if title in visited:
        continue

    visited.add(title)

    success = False

    for retry in range(MAX_RETRIES):

        try:

            page = download_article(title)

            if page is None:
                raise Exception("Page not found")

            filename = clean_filename(page["title"]) + ".txt"

            filepath = OUTPUT_DIR / filename

            if filepath.exists():

                success = True

                break

            with open(
                filepath,
                "w",
                encoding="utf-8"
            ) as f:

                f.write(page["text"])

            metadata.append({

                "Title": page["title"],
                "URL": page["url"],
                "Words": len(page["text"].split()),
                "Characters": len(page["text"]),
                "File": filename

            })

            success = True

            break

        except Exception as e:

            if retry == MAX_RETRIES - 1:

                errors.append({

                    "Title": title,
                    "Error": str(e)

                })

            time.sleep(SLEEP_TIME)

# =====================================================
# Save CSV
# =====================================================

metadata_df = pd.DataFrame(metadata)

metadata_df.to_csv(
    OUTPUT_DIR / "metadata.csv",
    index=False
)

errors_df = pd.DataFrame(errors)

errors_df.to_csv(
    OUTPUT_DIR / "error_log.csv",
    index=False
)

print("\n")
print("="*60)
print("Download Completed")
print("="*60)
print(f"Articles Downloaded : {len(metadata_df)}")
print(f"Failed Downloads    : {len(errors_df)}")
print(f"Folder              : {OUTPUT_DIR}")
print("="*60)

Searching Wikipedia

Searching : Artificial Intelligence
Found 10 articles

Searching : Machine Learning
Found 10 articles

Searching : Deep Learning
Found 10 articles

Searching : Large Language Model
Found 10 articles

Searching : Natural Language Processing
Found 10 articles

Searching : Data Science
Found 10 articles

Searching : Computer Vision
Found 10 articles

Searching : Generative AI
Found 10 articles

Searching : Transformer
Found 10 articles

Searching : Neural Network
Found 10 articles

Searching : Prompt Engineering
429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=Prompt+Engineering&srlimit=10&format=json

Searching : Retrieval Augmented Generation
429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=Retrieval+Augmented+Generation&srlimit=10&format=json

Searching : Semantic Search
429 Client Error: Too Many Requests for url: https://en.wikipedia.o

100%|██████████| 88/88 [09:27<00:00,  6.44s/it]



Download Completed
Articles Downloaded : 82
Failed Downloads    : 6
Folder              : AI_Knowledge_Base
